# Procesamiento de Datos

In [1]:
import pandas as pd
import numpy as np
import json
import re

## Museos México

In [2]:
museos = {'FridaKahlo':'data/museo_frida_kahlo/dataset.csv',
           'MUNAL':'data/museo_nacional_de_arte/dataset.csv', 
           'Soumaya': 'data/museo_soumaya_fundaci_n_carlos_slim/dataset.csv', 
           'Blaisten': 'data/colecci_n_blaisten/dataset.csv', 
           'MACAY': 'data/museo_de_arte_contempor_neo_ateneo_de_yucat_n_macay_fernando_garc_a_ponce/dataset.csv', 
           'Arocena': 'data/museo_arocena/dataset.csv'}

In [3]:
dataframes = {}
for name, filename in museos.items():
    dataframes[name] = pd.read_csv(filename)

In [4]:
dataframes.keys()

dict_keys(['FridaKahlo', 'MUNAL', 'Soumaya', 'Blaisten', 'MACAY', 'Arocena'])

In [5]:
common_all = set.intersection(*[set(df.columns) for df in dataframes.values()])
common_all

{'Creador',
 'Derechos',
 'Dimensiones físicas',
 'Título',
 'artista',
 'descripcion',
 'imagen',
 'link',
 'titulo'}

In [6]:
df1 = pd.concat(dataframes, names=["museo"])

In [7]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 1711 entries, ('FridaKahlo', np.int64(0)) to ('Arocena', np.int64(125))
Data columns (total 67 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   titulo                              1711 non-null   object
 1   artista                             1711 non-null   object
 2   link                                1711 non-null   object
 3   descripcion                         1184 non-null   object
 4   imagen                              1711 non-null   object
 5   Creador                             1402 non-null   object
 6   Derechos                            1221 non-null   object
 7   Dimensiones físicas                 1450 non-null   object
 8   Enlace externo                      396 non-null    object
 9   Fecha                               163 non-null    object
 10  Fecha de creación                   1348 non-null   object
 11  Firma de

In [8]:
print(df1.count().sort_values(ascending=False).to_string())

titulo                                1711
artista                               1711
link                                  1711
imagen                                1711
Título                                1711
Dimensiones físicas                   1450
Creador                               1402
Fecha de creación                     1348
Derechos                              1221
descripcion                           1184
Ayuda                                  771
Tipo                                   707
Manifestación artística                648
Técnica artística                      589
Enlace externo                         396
Tema representado                      390
Sexo del creador                       265
Vida del creador                       257
Palabras clave del asunto              255
Nacionalidad del creador               232
Lugar de nacimiento del creador        222
Persona representada                   208
Género artístico                       196
Lugar de fa

In [9]:
museums_with_column = []
for museum_name, df in dataframes.items():
    if 'Ayuda' in df.columns:
        museums_with_column.append(museum_name)

print(museums_with_column)


['Soumaya']


In [10]:
dataframes['Soumaya']['Ayuda'].value_counts()

Ayuda
Óleo sobre lienzo                                                                                                                                                        261
Óleo sobre tabla                                                                                                                                                          69
canvas                                                                                                                                                                    53
Óleo sobre lámina de cobre                                                                                                                                                15
Óleo sobre lámina de zinc                                                                                                                                                 12
                                                                                                                                 

In [11]:
def unify_priority_columns(df, new_col, cols_priority):
    cols = [c for c in cols_priority if c in df.columns]

    if not cols:
        return df

    df[new_col] = df[cols[0]]

    # rellena con las siguientes
    for col in cols[1:]:
        df[new_col] = df[new_col].fillna(df[col])

    return df

def unify_columns(df):
    df = unify_priority_columns(df, "titulo", [
        "titulo",
        "Título",
        "original title",
        "title (spanish)",
        "Original Spanish object note"
    ])
    df = unify_priority_columns(df, "descripcion", [
        "descripcion",
        "description"
    ])

    df = unify_priority_columns(df, "tipo_obra", [
        "Tipo",
        "Manifestación artística"
    ])


    df = unify_priority_columns(df, "tecnica", [
        "Técnica artística",
        "Ayuda"
    ])


    df = unify_priority_columns(df, "tema", [
        "Tema representado",
        "theme"
    ])

    df = unify_priority_columns(df, "genero_artistico", [
        "Género artístico",
        "Movimiento artístico"
    ])

  
    df = unify_priority_columns(df, "origen", [
        "Origen",
        "Lugar de creación",
        "Lugar"
    ])


    df = unify_priority_columns(df, "fecha", [
        "Fecha de creación",
        "Fecha"
    ])

    df = unify_priority_columns(df, "artista", [
        "artista",
        "Creador",
        "painter",
        "Sculptor",
        "Photographer",
        "Fotógrafo"
    ])

    cols_to_drop = [
        "Título", "original title", "title (spanish)", "Original Spanish object note",
        "description",
        "Tipo", "Manifestación artística",
        "Técnica artística", "Ayuda",
        "theme",
        "Movimiento artístico",
        "Lugar de creación", "Lugar",
        "Fecha",
        "Creador", "painter", "Sculptor", "Photographer", "Fotógrafo"
    ]

    cols_to_drop = [c for c in cols_to_drop if c in df.columns]

    df = df.drop(columns=cols_to_drop)

    return df

def clean_strings(df):
    def limpiar(x):
        if isinstance(x, str):
            return x.strip()
        return x

    return df.applymap(limpiar)
def add_museum_column(df):
    df = df.copy()
    df["museo"] = df.index.get_level_values(0)
    return df
def drop_sparse_columns(df, threshold=50):
    cols_to_drop = [col for col in df.columns if df[col].count() < threshold]
    return df.drop(columns=cols_to_drop)
def clean_pipeline(df):
    #print("ANTES:", df.shape)
    #print(df[["Fecha", "Fecha de creación"]].count())

    df = unify_columns(df)

    #print("DESPUÉS:", df.shape)
    #print(df["fecha"].count())
    df = clean_strings(df)
    df = add_museum_column(df)
    df = drop_sparse_columns(df, 70)

    return df

In [12]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 1711 entries, ('FridaKahlo', np.int64(0)) to ('Arocena', np.int64(125))
Data columns (total 67 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   titulo                              1711 non-null   object
 1   artista                             1711 non-null   object
 2   link                                1711 non-null   object
 3   descripcion                         1184 non-null   object
 4   imagen                              1711 non-null   object
 5   Creador                             1402 non-null   object
 6   Derechos                            1221 non-null   object
 7   Dimensiones físicas                 1450 non-null   object
 8   Enlace externo                      396 non-null    object
 9   Fecha                               163 non-null    object
 10  Fecha de creación                   1348 non-null   object
 11  Firma de

In [13]:
print(df1.shape)

(1711, 67)


In [14]:
df1 = clean_pipeline(df1)

C:\Users\roxan\AppData\Local\Temp\ipykernel_41300\2677936922.py:96: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(limpiar)


In [15]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 1711 entries, ('FridaKahlo', np.int64(0)) to ('Arocena', np.int64(125))
Data columns (total 29 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   titulo                              1711 non-null   object
 1   artista                             1711 non-null   object
 2   link                                1711 non-null   object
 3   descripcion                         1184 non-null   object
 4   imagen                              1711 non-null   object
 5   Derechos                            1221 non-null   object
 6   Dimensiones físicas                 1450 non-null   object
 7   Enlace externo                      396 non-null    object
 8   Fecha de creación                   1348 non-null   object
 9   Lugar de fallecimiento del creador  182 non-null    object
 10  Lugar de nacimiento del creador     222 non-null    object
 11  Nacional

In [16]:
df1.drop(columns=['Derechos', 'link', 'Enlace externo', 'Vida del creador', 'Original title', 'Provenance', 'Persona representada', 'Tema representado', 'Exhibición', 'Origen', 'Fecha de creación', 'Género artístico'], inplace=True)

In [17]:
def rename_columns(df):
    return df.rename(columns={
        "titulo": "title",
        "fecha": "date",
        "artista": "artist",
        "descripcion": "description",
        "imagen": "image_path",
        "tecnica": "technique",
        "Dimensiones físicas": "dimensions",
        "Nacionalidad del creador": "artist_nationality",
        "Sexo del creador": "artist_sex",
        "Lugar de nacimiento del creador": "artist_birthplace",
        "Lugar de fallecimiento del creador": "artist_deathplace",
        "genero_artistico": "genre",
        "Palabras clave del asunto": "keywords",
        "tipo_obra": "type",
        "tema": "theme",
        "museo": "museum",
        "origen": "origin"
    })

In [18]:
df1 = rename_columns(df1)

In [19]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 1711 entries, ('FridaKahlo', np.int64(0)) to ('Arocena', np.int64(125))
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   title               1711 non-null   object
 1   artist              1711 non-null   object
 2   description         1184 non-null   object
 3   image_path          1711 non-null   object
 4   dimensions          1450 non-null   object
 5   artist_deathplace   182 non-null    object
 6   artist_birthplace   222 non-null    object
 7   artist_nationality  232 non-null    object
 8   artist_sex          265 non-null    object
 9   keywords            255 non-null    object
 10  type                1355 non-null   object
 11  technique           1360 non-null   object
 12  theme               393 non-null    object
 13  genre               212 non-null    object
 14  origin              253 non-null    object
 15  date                1511

In [20]:
df1['dimensions'].value_counts()

dimensions
w50 x h65 cm (Complete)                                19
Formato rectangular vertical. Objeto bidimensional.    18
24.2x19.2cm                                            14
w56 x h76 cm (Complete)                                10
w76 x h56 cm (Complete)                                10
                                                       ..
H 55.5 x W: 23 x ? 19.8 CM                              1
w1622 x h1052 mm (complete)                             1
85 x 64 x 2 cm                                          1
H:53.5 X W:52.5 X D:6.5                                 1
63 x 26 x 20 cm                                         1
Name: count, Length: 1321, dtype: int64

## Normalizar Valores

### Dimensiones

In [21]:
def parse_dimensions(text):
    if not isinstance(text, str) or text.strip() == "":
        return np.nan, np.nan, np.nan

    text = text.lower()

    if "mm" in text:
        factor = 0.1 
    else:
        factor = 1  

    nums = re.findall(r"\d+\.?\d*", text)
    nums = [float(n) * factor for n in nums]

    if len(nums) == 0:
        return np.nan, np.nan, np.nan

    elif len(nums) == 1:
        return nums[0], np.nan, np.nan

    elif len(nums) == 2:
        return nums[0], nums[1], np.nan

    else:
        return nums[0], nums[1], nums[2]

In [22]:
df1[["width_cm", "height_cm", "depth_cm"]] = df1["dimensions"].apply(
    lambda x: pd.Series(parse_dimensions(x))
)

In [23]:
df1['width_cm'].value_counts()

width_cm
50.0     24
100.0    19
56.0     17
60.0     17
24.2     15
         ..
37.7      1
214.0     1
44.3      1
84.4      1
75.6      1
Name: count, Length: 747, dtype: int64

### Tipo

In [24]:
print(df1['type'].value_counts().to_string())

type
Pintura                                          738
Painting                                         136
PAINTING                                         111
Escultura                                         73
Fotografía                                        31
Photograph                                        21
Litografía                                        20
Acrílico sobre tela                               19
Collage                                           19
Serigrafía                                        17
pintura                                           16
Pintura al óleo.                                  12
Mixta sobre tela                                  11
Serigrafía/ Collage                               10
Pintura de panel / Óleo sobre madera               8
Pintura al óleo                                    8
Dibujo                                             7
sculpture                                          6
Óleo sobre tela                          

In [25]:
def normalize_type(text):
    if not isinstance(text, str):
        return np.nan

    text = text.lower().strip()

    if any(word in text for word in [
        "pintura", "painting", "óleo", "oleo", "acrílico"
    ]):
        return "pintura"

    if any(word in text for word in [
        "escultura", "sculpture", "relief", "talla", "Ecultura"
    ]):
        return "escultura"
    
    if any(word in text for word in [
        "fotografía", "photograph", "foto"
    ]):
        return "fotografía"

    if any(word in text for word in [
        "grabado", "litografía", "lithograph", "serigrafía"
    ]):
        return "impresiones"

    if any(word in text for word in [
        "dibujo", "drawing"
    ]):
        return "dibujo"

    if any(word in text for word in [
        "textil", "huipil", "rebozo", "vestido", "corsé"
    ]):
        return "textil"
    
    if any(word in text for word in [
        "mixta", "collage"
    ]):
        return "mixta"

    if any(word in text for word in [
        "collar", "gafas", "abanico", "bota", "estuche"
    ]):
        return "objeto"

    return "otro"

In [26]:
df1["type_norm"] = df1["type"].apply(normalize_type)

In [27]:
df1['type_norm'].value_counts()

type_norm
pintura        1063
escultura        86
fotografía       56
impresiones      56
mixta            46
otro             18
textil           13
dibujo           11
objeto            6
Name: count, dtype: int64

In [28]:
df1[df1['type_norm'] == 'otro']['type'].value_counts()

type
Ecultura                4
Orfebrerí­a             4
Artes decorativas       2
Aparato ortopédico      1
Zapatillas              1
Tocado                  1
Aparatos ortopédicos    1
Featherwork             1
Manuscript              1
Gouache sobre papel     1
Bordado                 1
Name: count, dtype: int64

### Fecha

In [29]:
print(df1['date'].value_counts().to_string())

date
1000                           41
Siglo XVIII                    40
1750/1800                      28
1976 - 1976                    19
1950                           19
1850/1850                      18
Siglo XVI                      16
Siglo XVII                     15
1850/1900                      15
1984 - 1984                    14
1750/1830                      13
1770/1800                      13
1922                           11
1985 - 1985                    11
1650/1650                      10
2018                           10
1840/1860                      10
1943                            9
1670/1730                       9
1947                            9
1977 - 1977                     9
1974 - 1974                     9
1986 - 1986                     9
1940                            9
1975 - 1975                     9
Siglo XIX                       8
1978 - 1978                     8
1939                            8
1670/1670                       8
1770/1830

In [30]:
def parse_date(text):
    if not isinstance(text, str):
        return np.nan, np.nan, False

    text = text.lower().strip()
    estimated = False

    if "ca" in text or "c." in text:
        estimated = True

    is_bc = "a.c" in text or "a. c" in text or "bc" in text


    match = re.search(r"siglo\s+([ivxlcdm]+)", text)
    if match:
        roman = match.group(1)
        roman_map = {'i':1,'v':5,'x':10,'l':50,'c':100,'d':500,'m':1000}

        def roman_to_int(s):
            total, prev = 0, 0
            for char in reversed(s):
                val = roman_map[char]
                if val < prev:
                    total -= val
                else:
                    total += val
                prev = val
            return total

        century = roman_to_int(roman)
        start = (century - 1) * 100
        end = century * 100

        return start, end, True


    nums = re.findall(r"\d{3,4}", text)

    if not nums:
        return np.nan, np.nan, estimated

    nums = [int(n) for n in nums]

    if is_bc:
        nums = [-n for n in nums]


    start = min(nums)
    end = max(nums)

    if len(nums) == 1:
        return start, start, estimated

    return start, end, estimated

In [31]:
df1[["year_start", "year_end", "estimated"]] = df1["date"].apply(
    lambda x: pd.Series(parse_date(x))
)

In [32]:
print(df1['year_end'].value_counts().to_string())

year_end
 1800.0    95
 1700.0    42
 1000.0    41
 1900.0    41
 1850.0    28
 1830.0    26
 1950.0    23
 1730.0    22
 1750.0    22
 1600.0    20
 1740.0    20
 1976.0    19
 1760.0    19
 1650.0    18
 1780.0    17
 1860.0    15
 1984.0    14
 1940.0    13
 1947.0    12
 1922.0    12
 1660.0    12
 1770.0    12
 1948.0    11
 1970.0    11
 1985.0    11
 1930.0    11
 1610.0    11
 1540.0    10
 1550.0    10
 1943.0    10
 1977.0    10
 1945.0    10
 1938.0    10
 1974.0    10
 2018.0    10
 1775.0     9
 1670.0     9
 1986.0     9
 1975.0     9
 1926.0     8
 1880.0     8
 1928.0     8
 1982.0     8
 1939.0     8
 1925.0     8
 1978.0     8
 1500.0     8
 1910.0     8
 1913.0     8
 1851.0     7
 1630.0     7
 1918.0     7
 1924.0     7
 1625.0     7
 1911.0     6
 1865.0     6
 1953.0     6
 1949.0     6
 1942.0     6
 1919.0     6
 1903.0     6
 1946.0     6
 1640.0     6
 1875.0     6
 1631.0     6
 1867.0     6
 1941.0     6
 1944.0     6
 1857.0     5
 1937.0     5
 1929.0    

### Sexo

In [33]:
print(df1['artist_sex'].value_counts().to_string())

artist_sex
Male         136
Masculino    119
Female        10


In [34]:
df1['artist_sex'] = df1['artist_sex'].replace({
    'Masculino': 'Male',
    'Femenino': 'Female'
})


In [35]:
df1['artist_sex'].value_counts()

artist_sex
Male      255
Female     10
Name: count, dtype: int64

### Technique

In [36]:
df1['technique'].value_counts()

technique
Óleo sobre lienzo                                                                                                 261
Óleo sobre tela                                                                                                   160
Oil on canvas                                                                                                      82
Óleo sobre tabla                                                                                                   71
canvas                                                                                                             53
                                                                                                                 ... 
Óleo y temple sobre tabla.                                                                                          1
Temple y oro sobre tabla. Marco de madera tallada y esgrafiada, con policromía y con aplicación de hoja de oro      1
Óleo sobre lámina de cobre. Marco de madera ta

In [37]:
def normalize_text(text):
    if not isinstance(text, str):
        return text
    
    text = text.lower().strip()

    replacements = {
        "á":"a","é":"e","í":"i","ó":"o","ú":"u","ñ":"n"
    }
    for k,v in replacements.items():
        text = text.replace(k,v)
    
    return text

In [38]:
TECHNIQUE_MAP = {
    "oleo": "oleo",
    "oil": "oleo",
    "acrílico": "acrilico",
    "acrilico": "acrilico",
    "acuarela": "acuarela",
    "watercolor": "acuarela",
    "gouache": "gouache",
    "tinta": "tinta",
    "grafito": "grafito",
    "pastel": "pastel",
    "litografia": "litografia",
    "serigrafia": "serigrafia",
    "temple": "temple"
}

SUPPORT_MAP = {
    "lienzo": "lienzo",
    "canvas": "lienzo",
    "tela": "tela",
    "papel": "papel",
    "carton": "carton",
    "madera": "madera",
    "tabla": "madera",
    "cobre": "metal",
    "zinc": "metal",
    "metal": "metal",
    "vidrio": "vidrio",
    "marfil": "marfil"
}

In [39]:
def extract_technique_support(text):
    if not isinstance(text, str):
        return None, None
    
    text = normalize_text(text)
    
    technique = None
    support = None
    
    for key in TECHNIQUE_MAP:
        if key in text:
            technique = TECHNIQUE_MAP[key]
            break
    
    for key in SUPPORT_MAP:
        if key in text:
            support = SUPPORT_MAP[key]
            break
    
    return technique, support

In [40]:
df1[["technique_clean", "support"]] = df1["technique"].apply(
    lambda x: pd.Series(extract_technique_support(x))
)

In [41]:
df1['support'].value_counts()

support
lienzo    435
tela      212
madera    209
papel      90
metal      76
marfil     26
carton     23
vidrio     13
Name: count, dtype: int64

In [42]:
df1['technique_clean'].value_counts()

technique_clean
oleo          851
acuarela       48
gouache        31
temple         27
tinta          26
pastel          7
litografia      2
grafito         2
Name: count, dtype: int64

### Theme

In [43]:
print(df1['theme'].value_counts().to_string())

theme
Paisaje                                                                                                                        69
Utensilios                                                                                                                     29
Herrería                                                                                                                       20
Numismática                                                                                                                    16
Virgen                                                                                                                         13
Santos                                                                                                                         10
batalla, caballo                                                                                                                8
Reliquias                                                                           

In [44]:
THEME_CATEGORIES = {
    "religion": [
        "virgen", "cristo", "crucifixion", "calvario", "piedad",
        "santos", "angel", "natividad", "epifania",
        "eucaristia", "sagrada familia", "juicio final",
        "resurreccion", "iglesia", "biblia"
    ],
    
    "paisaje": [
        "paisaje", "rio", "montana", "bosque", "nubes",
        "mar", "costa", "volcan", "ruinas", "campo"
    ],
    
    "retrato": [
        "retrato", "autorretrato", "anciana", "anciano",
        "marquesa", "monja", "persona"
    ],
    
    "vida_cotidiana": [
        "campesinos", "boda", "festividad", "mercado",
        "trabajo", "cocina", "costura", "familia"
    ],
    
    "naturaleza_muerta": [
        "bodegon", "fruta", "comida", "mesa", "utensilios"
    ],
    
    "mitologia": [
        "mitologia", "troya", "agamenon", "ninfas", "satiros"
    ],
    
    "arquitectura": [
        "arquitectura", "edificio", "coliseo", "puente",
        "ciudad", "plaza", "torre"
    ],
    
    "animales": [
        "caballo", "perro", "vaca", "oveja", "leon",
        "aves", "animal"
    ],
    
    "guerra": [
        "batalla", "soldados", "ejercito", "caballeria"
    ]
}

In [45]:
def classify_theme(text):
    if not isinstance(text, str):
        return None
    
    text = normalize_text(text)
    
    matches = []
    
    for category, keywords in THEME_CATEGORIES.items():
        for word in keywords:
            if word in text:
                matches.append(category)
                break
    
    if not matches:
        return "otro"
    
    return list(set(matches))

In [46]:
df1["theme_tag"] = df1["theme"].apply(classify_theme)

In [47]:
df1['theme_tag'].value_counts()

theme_tag
otro                                  147
[paisaje]                              86
[religion]                             50
[naturaleza_muerta]                    37
[animales]                             11
[arquitectura]                          8
[animales, guerra]                      8
[paisaje, arquitectura]                 8
[vida_cotidiana]                        8
[retrato]                               6
[religion, paisaje]                     4
[retrato, arquitectura]                 2
[animales, paisaje]                     2
[mitologia]                             2
[mitologia, paisaje]                    2
[religion, animales]                    2
[animales, mitologia]                   2
[religion, animales, arquitectura]      1
[animales, arquitectura]                1
[mitologia, guerra]                     1
[retrato, paisaje]                      1
[religion, vida_cotidiana]              1
[guerra]                                1
[vida_cotidiana, arquite

### Lugar

In [48]:
print(  df1['origin'].value_counts().to_string())

origin
Colección privada                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [49]:
def clean_origin_text(x):
    if not isinstance(x, str):
        return None
    
    x = x.strip()
    
    if len(x) > 100:
        return None
    
    return x

In [50]:
def extract_region_country(x):
    if not isinstance(x, str):
        return None, None
    
    parts = [p.strip() for p in x.split(",") if p.strip()]
    
    if len(parts) == 0:
        return None, None
    
    elif len(parts) == 1:
        return None, parts[0]
    
    else:
        return parts[-2], parts[-1] 

In [51]:
def normalize_country(country):
    if not isinstance(country, str):
        return None
    
    country = country.lower()
    
    mapping = {
        "mexico": "México",
        "méxico": "México",
        "españa": "España",
        "spain": "España",
        "francia": "Francia",
        "france": "Francia",
        "italia": "Italia",
        "new york": "Estados Unidos",
        "usa": "Estados Unidos",
        "estados unidos": "Estados Unidos",
        "europa": "Europa",
        "nueva españa": "Nueva España",
        "nueva españa (méxico)": "Nueva España"
    }
    
    return mapping.get(country, country.capitalize())

In [52]:
def process_origin(df):
    df["origin"] = df["origin"].apply(clean_origin_text)
    
    regiones = []
    paises = []
    
    for val in df["origin"]:
        region, country = extract_region_country(val)
        
        region = region.strip() if isinstance(region, str) else None
        country = normalize_country(country)
        
        regiones.append(region)
        paises.append(country)
    
    df["region"] = regiones
    df["pais"] = paises
    
    return df

In [53]:
df1 = process_origin(df1)

In [54]:
def clean_nationality(text):
    if not isinstance(text, str):
        return None
    
    text = text.lower().strip()

    # eliminar ruido común
    text = text.split("born")[0]
    text = text.split(" or ")[0]
    text = text.split(" and ")[0]
    text = text.split(",")[0]

    text = text.strip()

    # diccionario de normalización
    mapping = {
        "italian": "italiano",
        "german": "alemán",
        "spanish": "español",
        "american": "estadounidense",
        "mexican": "mexicano",
        "hungarian": "húngaro",
    }

    return mapping.get(text, text)

In [55]:
df1["artist_nationality"] = df1["artist_nationality"].apply(clean_nationality)

In [56]:
print(df1['artist_nationality'].value_counts().to_string())

artist_nationality
mexicano          202
español            17
alemán             10
húngaro             1
estadounidense      1
italiano            1


## The Met

In [57]:
with open("data/the_metropolitan_museum_of_art/dataset.json", "r", encoding="utf-8") as f:
    met_data = json.load(f)

met = pd.DataFrame(met_data)

In [58]:
met.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1207 entries, 0 to 1206
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   objectID           1207 non-null   int64 
 1   title              1207 non-null   object
 2   artist             1207 non-null   object
 3   artistNationality  1207 non-null   object
 4   objectDate         1207 non-null   object
 5   medium             1207 non-null   object
 6   dimensions         1207 non-null   object
 7   department         1207 non-null   object
 8   culture            1207 non-null   object
 9   country            1207 non-null   object
 10  classification     1207 non-null   object
 11  image              1207 non-null   object
 12  objectURL          1207 non-null   object
dtypes: int64(1), object(12)
memory usage: 122.7+ KB


In [59]:
met.head()

,objectID,title,artist,artistNationality,objectDate,medium,dimensions,department,culture,country,classification,image,objectURL
0,437056,Tommaso di Folco Portinari (1428–1501); Maria ...,Hans Memling,Netherlandish,ca. 1470,Oil on oak,"(.626, Tommaso) overall 17 3/8 x 13 1/4 in. (4...",European Paintings,,,Paintings,https://images.metmuseum.org/CRDImages/ep/orig...,https://www.metmuseum.org/art/collection/searc...
1,929079,Orchids and Rocks,Yi Ha-eung,Korean,1888,Hanging scroll; ink on silk,Image: 60 1/2 × 40 1/16 in. (153.7 × 101.8 cm)...,Asian Art,Korea,,Paintings,https://images.metmuseum.org/CRDImages/as/orig...,https://www.metmuseum.org/art/collection/searc...
2,764091,Our Lady of Valvanera,Unknown,,ca. 1770–80,Oil and gold on canvas,80 1/16 × 95 7/8 in. (203.4 × 243.5 cm),The American Wing,Peru (Cuzco),,,https://images.metmuseum.org/CRDImages/ad/orig...,https://www.metmuseum.org/art/collection/searc...
3,453250,Study of a Bird,Riza-yi 'Abbasi,Iranian,dated 1043 AH/1634 CE,"Ink, opaque watercolor, gold, and silver on paper",Overall:\r\n H. 4 1/4 in. (10.8 cm...,Islamic Art,,Iran,Codices,https://images.metmuseum.org/CRDImages/is/orig...,https://www.metmuseum.org/art/collection/searc...
4,764095,"Christ Carrying the Cross, called ""The Lord of...",Unknown,,ca. 1770–75,Oil and gold on canvas,65 × 48 in. (165.1 × 121.9 cm),The American Wing,Peru (Cuzco),,,https://images.metmuseum.org/CRDImages/ad/orig...,https://www.metmuseum.org/art/collection/searc...


In [60]:
def normalize_missing(df):
    df = df.replace({
        None: np.nan,
        "": np.nan,
        " ": np.nan
    })
    return df

In [61]:
normalize_missing(met)

,objectID,title,artist,artistNationality,objectDate,medium,dimensions,department,culture,country,classification,image,objectURL
0,437056,Tommaso di Folco Portinari (1428–1501); Maria ...,Hans Memling,Netherlandish,ca. 1470,Oil on oak,"(.626, Tommaso) overall 17 3/8 x 13 1/4 in. (4...",European Paintings,NaN,NaN,Paintings,https://images.metmuseum.org/CRDImages/ep/orig...,https://www.metmuseum.org/art/collection/searc...
1,929079,Orchids and Rocks,Yi Ha-eung,Korean,1888,Hanging scroll; ink on silk,Image: 60 1/2 × 40 1/16 in. (153.7 × 101.8 cm)...,Asian Art,Korea,NaN,Paintings,https://images.metmuseum.org/CRDImages/as/orig...,https://www.metmuseum.org/art/collection/searc...
2,764091,Our Lady of Valvanera,Unknown,NaN,ca. 1770–80,Oil and gold on canvas,80 1/16 × 95 7/8 in. (203.4 × 243.5 cm),The American Wing,Peru (Cuzco),NaN,NaN,https://images.metmuseum.org/CRDImages/ad/orig...,https://www.metmuseum.org/art/collection/searc...
3,453250,Study of a Bird,Riza-yi 'Abbasi,Iranian,dated 1043 AH/1634 CE,"Ink, opaque watercolor, gold, and silver on paper",Overall:\r\n H. 4 1/4 in. (10.8 cm...,Islamic Art,NaN,Iran,Codices,https://images.metmuseum.org/CRDImages/is/orig...,https://www.metmuseum.org/art/collection/searc...
4,764095,"Christ Carrying the Cross, called ""The Lord of...",Unknown,NaN,ca. 1770–75,Oil and gold on canvas,65 × 48 in. (165.1 × 121.9 cm),The American Wing,Peru (Cuzco),NaN,NaN,https://images.metmuseum.org/CRDImages/ad/orig...,https://www.metmuseum.org/art/collection/searc...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1202,53213,Herd Boy with Ox,Kano Masanobu 狩野正信,Japanese,late 15th century,Folding fan mounted as a hanging scroll; ink a...,Image: 8 7/8 × 18 1/2 in. (22.5 × 47 cm)\r\nOv...,Asian Art,Japan,NaN,Paintings,https://images.metmuseum.org/CRDImages/as/orig...,https://www.metmuseum.org/art/collection/searc...
1203,203915,Seated Faun,"Andrea Briosco, called Riccio",Italian,ca. 1540–50,Bronze,Overall (confirmed): 11 1/4 × 7 × 7 1/2 in. (2...,European Sculpture and Decorative Arts,NaN,NaN,Sculpture-Bronze,https://images.metmuseum.org/CRDImages/es/orig...,https://www.metmuseum.org/art/collection/searc...
1204,436523,The Adoration of the Magi,Hugo van der Goes,Netherlandish,NaN,Oil on wood,29 1/8 x 25 5/8 in. (74 x 65.1 cm),European Paintings,NaN,NaN,Paintings,https://images.metmuseum.org/CRDImages/ep/orig...,https://www.metmuseum.org/art/collection/searc...
1205,436553,A City on a Rock,Goya,Spanish,NaN,Oil on canvas,33 × 41 in. (83.8 × 104.1 cm),European Paintings,NaN,NaN,Paintings,https://images.metmuseum.org/CRDImages/ep/orig...,https://www.metmuseum.org/art/collection/searc...


In [62]:
def transform_met(df):
    return pd.DataFrame({
        "title": df["title"],
        "artist": df["artist"],
        "description": np.nan,

        "image_path": df["image"],
        "dimensions": df["dimensions"],

        "artist_nationality": df["artistNationality"],
        "artist_birthplace": np.nan,
        "artist_deathplace": np.nan,
        "artist_sex": np.nan,

        "type": df["classification"],
        "type_norm": df["classification"],

        "technique": df["medium"],

        "theme": np.nan,
        "genre": np.nan,
        "keywords": np.nan,

        "origin": df["country"],
        "pais": df["country"],

        "date": df["objectDate"],
        "museum": "MET"
    })



## Normalizar Valores

### Imágenes

In [63]:
def limpiar_nombre(nombre):
    if not isinstance(nombre, str):
        return "unknown"

    nombre = unicodedata.normalize('NFKD', nombre)
    nombre = nombre.encode('ascii', 'ignore').decode('ascii')

    nombre = nombre.lower()
    nombre = nombre.replace(" ", "_")

    nombre = re.sub(r'[^a-z0-9_]', '', nombre)
    nombre = re.sub(r'_+', '_', nombre)  # 👈 evita ___

    return nombre.strip("_")

def generar_nombre_imagen(row):
    titulo = limpiar_nombre(row["title"])
    object_id = row["objectID"]

    return f"{titulo}_{object_id}.jpg"


def generar_ruta_imagen(row):
    nombre = generar_nombre_imagen(row)
    return f"data/the_metropolitan_museum_of_art/_images/{nombre}"

### Fecha

In [64]:
def parse_dates_met(df):
    parsed = df["date"].apply(lambda x: pd.Series(parse_date(x)))
    parsed.columns = ["year_start", "year_end", "estimated"]

    return pd.concat([df, parsed], axis=1)


### Dimensiones

In [65]:

def parse_dimensions_met(text):
    if not isinstance(text, str):
        return np.nan, np.nan, np.nan

    match = re.search(r"\(([\d\.]+)\s*[×x]\s*([\d\.]+)\s*cm\)", text)

    if match:
        height = float(match.group(1))
        width = float(match.group(2))
        return width, height, np.nan

    return np.nan, np.nan, np.nan


### Tipo

In [66]:

def normalize_type_es(text):
    if not isinstance(text, str) or text.strip() == "":
        return np.nan
    
    text = text.lower()

    if "painting" in text or "oil" in text:
        return "pintura"

    if "drawing" in text or "pastel" in text:
        return "dibujo"

    if "print" in text or "etching" in text or "engraving" in text:
        return "impresión"

    if "photograph" in text:
        return "fotografía"
    
    if any(x in text for x in [
        "sculpture", "stone", "marble", "jade", "ivories", "wood"
    ]):
        return "escultura"

    if "textile" in text or "tapestry" in text or "rug" in text:
        return "textil"
    
    if any(x in text for x in [
        "book", "manuscript", "codices", "calligraphy", "illustrated"
    ]):
        return "libro"

    if any(x in text for x in [
        "ceramic", "glass", "metal", "jewelry", "horology",
        "weapon", "armor", "instrument", "shell", "lacquer"
    ]):
        return "objeto"

    return "otro"


### Lugar

In [74]:

def normalize_pais(text):
    if not isinstance(text, str):
        return np.nan
    
    text = text.lower()

    if "mexico" in text:
        return "México"
    if "peru" in text:
        return "Perú"
    if "guatemala" in text:
        return "Guatemala"
    if "spain" in text:
        return "España"
    if "france" in text:
        return "Francia"
    if "italy" in text:
        return "Italia"
    if "germany" in text:
        return "Alemania"
    if "netherlands" in text:
        return "Países Bajos"
    if "japan" in text:
        return "Japón"
    if "china" in text:
        return "China"
    if "india" in text:
        return "India"
    if "iran" in text:
        return "Irán"
    if "egypt" in text:
        return "Egipto"
    if "united states" in text or "american" in text:
        return "Estados Unidos"
    if "afghanistan" in text:
        return "Afganistán"
    if "pakistan" in text:
        return "Pakistán"
    if "syria" in text:
        return "Siria"
    if "iraq" in text:
        return "Irak"
    if "england" in text or "united kingdom" in text:
        return "Reino Unido"
    if "belgium" in text:
        return "Bélgica"
    if "ethiopia" in text:
        return "Etiopía"
    if "solomon islands" in text:
        return "Islas Salomón"
    if "turkey" in text:
        return "Turquía"
    if "papua new guinea" in text:
        return "Papúa Nueva Guinea"
    if "nubia" in text:
        return "Nubia"
    if "new caledonia" in text:
        return "Nueva Caledonia"
    if "uzbekistan" in text:
        return "Uzbekistán"
    if "panama" in text:
        return "Panamá"
    if "costa rica" in text:
        return "Costa Rica"
    if "nepal" in text:
        return "Nepal"

    return "otro"


In [75]:

def clean_nationality(text):
    if not isinstance(text, str):
        return None
    
    text = text.lower().strip()

    # eliminar ruido común
    text = text.split("born")[0]
    text = text.split(" or ")[0]
    text = text.split(" and ")[0]
    text = text.split(",")[0]

    text = text.strip()

    # diccionario de normalización
    mapping = {
        "french": "francés",
        "italian": "italiano",
        "netherlandish": "neerlandés",
        "dutch": "neerlandés",
        "flemish": "flamenco",
        "german": "alemán",
        "british": "británico",
        "english": "británico",
        "spanish": "español",
        "american": "estadounidense",
        "mexican": "mexicano",
        "japanese": "japonés",
        "chinese": "chino",
        "korean": "coreano",
        "indian": "indio",
        "iranian": "iraní",
        "egyptian": "egipcio",
        "greek": "griego",
        "swiss": "suizo",
        "austrian": "austriaco",
        "danish": "danés",
        "norwegian": "noruego",
        "belgian": "belga",
        "bohemian": "bohemio",
        "ethiopian": "etíope",
        "guatemalan": "guatemalteco",
        "african": "africano",
        "khitan": "khitan",  # lo puedes dejar como histórico
        "northern european": "europeo"
    }

    return mapping.get(text, text)  # fallback: deja lo que no reconozca


### Técnica

In [76]:
def normalize_technique(text):
    if not isinstance(text, str):
        return None
    
    t = text.lower()

    if any(k in t for k in ["oil"]):
        return "óleo"
    if any(k in t for k in ["ink"]):
        return "tinta"
    if any(k in t for k in ["watercolor", "gouache"]):
        return "acuarela"
    if "tempera" in t:
        return "temple"
    if "pastel" in t:
        return "pastel"
    if any(k in t for k in ["etching", "engraving", "mezzotint", "drypoint"]):
        return "grabado"
    if "lithograph" in t:
        return "litografía"
    if any(k in t for k in ["photo", "print", "albumen"]):
        return "fotografía"
    if any(k in t for k in ["ceramic", "porcelain", "earthenware"]):
        return "cerámica"
    if any(k in t for k in ["textile", "silk", "wool"]):
        return "textil"
    
    return "otro"


In [77]:
def normalize_support(text):
    if not isinstance(text, str):
        return None
    
    t = text.lower()

    if "canvas" in t:
        return "lienzo"
    if any(k in t for k in ["paper", "cardboard"]):
        return "papel"
    if any(k in t for k in ["wood", "panel", "oak", "walnut", 'oak', 'linden', 'spruce', 'walnut', 'pine', 'fir']):
        return "madera"
    if any(k in t for k in ["metal", "bronze", "copper", "silver", "gold", "brass", "steel"]):
        return "metal"
    if "silk" in t:
        return "seda"
    if "linen" in t:
        return "lino"
    if any(k in t for k in ["ceramic", "porcelain"]):
        return "cerámica"
    if any(k in t for k in ["stone", "marble", "alabaster"]):
        return "piedra"
    if "glass" in t:
        return "vidrio"
    if "ivory" in t:
        return "marfil"
    if "parchment" in t or "vellum" in t:
        return "pergamino"

    return "otro"

In [81]:

def split_technique(text):
    if not isinstance(text, str):
        return None, None
    
    text = text.lower()

    if " on " in text:
        parts = text.split(" on ")
        return parts[0], parts[1]

    return text, None
def clean_technique_met(df):
    split = df["technique"].apply(lambda x: pd.Series(split_technique(x)))
    split.columns = ["technique_raw", "support_raw"]

    df = pd.concat([df, split], axis=1)

    df["technique_clean"] = df["technique_raw"].apply(normalize_technique)
    df["support"] = df["support_raw"].apply(normalize_support)

    return df


### Pipeline

In [82]:


def clean_met_pipeline(df):
    df = transform_met(df)

    df = clean_strings(df)

    df = parse_dates_met(df)

    dims = df["dimensions"].apply(lambda x: pd.Series(parse_dimensions_met(x)))
    dims.columns = ["width_cm", "height_cm", "depth_cm"]

    df = pd.concat([df, dims], axis=1)

    df = clean_technique_met(df)

    df = df.replace(r"^\s*$", np.nan, regex=True)
    df["type_norm"] = df["type"].apply(normalize_type)
    df["pais"] = df["pais"].apply(normalize_pais)
    df["artist_nationality"] = df["artist_nationality"].apply(clean_nationality)

    return df


In [83]:
met = clean_met_pipeline(met)

C:\Users\roxan\AppData\Local\Temp\ipykernel_41300\2677936922.py:96: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(limpiar)


In [84]:
met.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1207 entries, 0 to 1206
Data columns (total 29 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   title               1207 non-null   object 
 1   artist              938 non-null    object 
 2   description         0 non-null      float64
 3   image_path          1207 non-null   object 
 4   dimensions          1197 non-null   object 
 5   artist_nationality  837 non-null    object 
 6   artist_birthplace   0 non-null      float64
 7   artist_deathplace   0 non-null      float64
 8   artist_sex          0 non-null      float64
 9   type                1146 non-null   object 
 10  type_norm           1146 non-null   object 
 11  technique           1207 non-null   object 
 12  theme               0 non-null      float64
 13  genre               0 non-null      float64
 14  keywords            0 non-null      float64
 15  origin              272 non-null    object 
 16  pais  

In [85]:
met.loc[met['type_norm'] == 'otro', 'type'].unique()

array(['Codices', 'Glass-Stained', 'Prints', 'Enamels-Cloisonné', 'Glass',
       'Ceramics', 'Ceramics-Porcelain-Export',
       'Manuscripts and Illuminations', 'Ceramics-Porcelain', 'Jade',
       'Woodwork', 'Chordophone-Zither-struck-piano', 'Horology', 'Wood',
       'Woodwork-Furniture', 'Ceramics-Containers', 'Jewelry',
       'Ivories-Elephant', 'Mosaics', 'Calligraphy', 'Stone-Implements',
       'Woodwork-Lacquer', 'Ivories', 'Ceramics-Vessels',
       'Enamels-Champlevé', 'Wallpaper', 'Wood-Containers',
       'Faience-Cylinder Seals', 'Pastels & Oil Sketches on Paper',
       'Stone', 'Wood-Architectural', 'Bone', 'Miniatures',
       'Hide-Documents', 'Vases', 'Metal-Ornaments', 'Amber', 'Screens',
       'Ceramics-Tiles', 'Ceramics-Pottery', 'Stone-Ornaments',
       'Illustrated Books', 'Medals', 'Ephemera', 'Shell-Ornaments',
       'Idiophone', 'Seals', 'Metalwork', 'Leatherwork',
       'Metalwork-Silver', 'Enamels-Painted', 'Metal', 'Lacquer', 'Books',
       'Knive

In [86]:
met.loc[met['technique_clean'] == 'other', 'technique_raw'].unique()

array([], dtype=object)

In [87]:
met.loc[met['pais'] == 'otro', 'origin'].unique()

array([], dtype=object)

In [88]:
print(met['artist_nationality'].value_counts().to_string())

artist_nationality
francés           175
italiano          154
neerlandés        151
japonés            63
británico          61
alemán             54
chino              43
iraní              37
estadounidense     24
español            23
indio              10
flamenco            8
austriaco           6
griego              5
suizo               5
bohemio             3
noruego             3
belga               2
mexicano            2
coreano             1
danés               1
egipcio             1
europeo             1
guatemalteco        1
etíope              1
africano            1
khitan              1


In [89]:
print(met['pais'].value_counts().to_string())

pais
Irán                  66
India                 39
Francia               25
Egipto                25
México                21
Perú                  18
Guatemala             13
Alemania               7
Etiopía                6
Siria                  6
Reino Unido            6
Italia                 5
Turquía                5
Estados Unidos         4
Nepal                  4
Pakistán               3
España                 3
Afganistán             3
Panamá                 2
Países Bajos           2
Nueva Caledonia        2
Islas Salomón          1
Nubia                  1
Papúa Nueva Guinea     1
Irak                   1
Uzbekistán             1
Costa Rica             1
Bélgica                1


## Merge

In [90]:
# cols_final = [
#     "title", "artist", "description", "image_path", "dimensions",
#     "artist_deathplace", "artist_birthplace", "artist_nationality", "artist_sex",
#     "keywords", "type", "type_norm", "technique", "theme", "genre",
#     "origin", "date", "museum",
#     "width_cm", "height_cm", "depth_cm",
#     "year_start", "year_end", "estimated",
#     "technique_clean", "support", "theme_tag",
#     "region", "pais"
# ]

# met = met.reindex(columns=cols_final)

In [91]:
df1 = df1.rename(columns={"theme_tag": "theme"})

In [92]:
cols_final = [
    "title", "artist", "description", "image_path", "dimensions",
    "artist_deathplace", "artist_birthplace", "artist_nationality", "artist_sex",
    "keywords",
    
    "type_norm",
    "technique_clean", "support",
    "theme",
    "pais",
    
    "year_start", "year_end", "estimated",
    "width_cm", "height_cm", "depth_cm",
    
    "museum"
]

In [93]:
df1 = df1[cols_final]

In [94]:
met = met[cols_final]

In [95]:
df1 = df1.rename(columns={
    "type_norm": "type",
    "technique_clean": "technique",
    "pais": "country"
})

In [96]:
met = met.rename(columns={
    "type_norm": "type",
    "technique_clean": "technique",
    "pais": "country"
})

In [97]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 1711 entries, ('FridaKahlo', np.int64(0)) to ('Arocena', np.int64(125))
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   title               1711 non-null   object 
 1   artist              1711 non-null   object 
 2   description         1184 non-null   object 
 3   image_path          1711 non-null   object 
 4   dimensions          1450 non-null   object 
 5   artist_deathplace   182 non-null    object 
 6   artist_birthplace   222 non-null    object 
 7   artist_nationality  232 non-null    object 
 8   artist_sex          265 non-null    object 
 9   keywords            255 non-null    object 
 10  type                1355 non-null   object 
 11  technique           994 non-null    object 
 12  support             1084 non-null   object 
 13  theme               393 non-null    object 
 14  theme               393 non-null    object 
 15  country

In [98]:
df1 = df1.loc[:, ~df1.columns.duplicated()]

In [99]:
met.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1207 entries, 0 to 1206
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   title               1207 non-null   object 
 1   artist              938 non-null    object 
 2   description         0 non-null      float64
 3   image_path          1207 non-null   object 
 4   dimensions          1197 non-null   object 
 5   artist_deathplace   0 non-null      float64
 6   artist_birthplace   0 non-null      float64
 7   artist_nationality  837 non-null    object 
 8   artist_sex          0 non-null      float64
 9   keywords            0 non-null      float64
 10  type                1146 non-null   object 
 11  technique           1207 non-null   object 
 12  support             783 non-null    object 
 13  theme               0 non-null      float64
 14  country             272 non-null    object 
 15  year_start          932 non-null    float64
 16  year_e

In [100]:
df_final = pd.concat([df1, met], axis=0)

In [101]:
df_final

,title,artist,description,image_path,dimensions,artist_deathplace,artist_birthplace,artist_nationality,artist_sex,keywords,...,support,theme,country,year_start,year_end,estimated,width_cm,height_cm,depth_cm,museum
"(FridaKahlo, 0)","Matilde, Adriana, Frida y Cristina Kahlo",Guillermo Kahlo,"Desde niña, Frida estuvo cerca de la fotografí...",data/museo_frida_kahlo/images/matilde_adriana_...,NaN,"Mexico City, Mexico","Baden, Germany",alemán,Male,NaN,...,None,NaN,None,1916.0,1916.0,False,NaN,NaN,NaN,FridaKahlo
"(FridaKahlo, 1)",Retrato de Agustín M Olmedo,Unknown,"Retrato de Agustín M. Olmedo, ca. 1927. Óleo s...",data/museo_frida_kahlo/images/retrato_de_agust...,NaN,NaN,NaN,None,NaN,NaN,...,None,NaN,None,1927.0,1927.0,False,NaN,NaN,NaN,FridaKahlo
"(FridaKahlo, 2)",El marxismo dará salud a los enfermos,Frida Kahlo,El estado de ánimo de Frida Kahlo se evidencia...,data/museo_frida_kahlo/images/el_marxismo_dar_...,w600 x h760 mm (without frame),"Mexico City, Mexico","Mexico City, Mexico",mexicano,Female,NaN,...,None,NaN,México,1954.0,1954.0,False,60.0,76.0,NaN,FridaKahlo
"(FridaKahlo, 3)",Viva la vida,Frida Kahlo,"""Viva la vida"" es un cuadro muy importante, ya...",data/museo_frida_kahlo/images/viva_la_vida_bAG...,w720 x h520 mm (with frame),"Mexico City, Mexico","Mexico City, Mexico",mexicano,Female,NaN,...,None,NaN,None,1954.0,1954.0,False,72.0,52.0,NaN,FridaKahlo
"(FridaKahlo, 4)",Las apariencias engañan,Frida Kahlo,"Este dibujo no se conoció hasta 2007, dado que...",data/museo_frida_kahlo/images/las_apariencias_...,w297 x h298 mm (without frame),"Mexico City, Mexico","Mexico City, Mexico",mexicano,Female,NaN,...,None,NaN,None,1934.0,1934.0,False,29.7,29.8,NaN,FridaKahlo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1202,Herd Boy with Ox,Kano Masanobu 狩野正信,NaN,https://images.metmuseum.org/CRDImages/as/orig...,Image: 8 7/8 × 18 1/2 in. (22.5 × 47 cm)\r\nOv...,NaN,NaN,japonés,NaN,NaN,...,papel,NaN,NaN,NaN,NaN,False,47.0,22.5,NaN,MET
1203,Seated Faun,"Andrea Briosco, called Riccio",NaN,https://images.metmuseum.org/CRDImages/es/orig...,Overall (confirmed): 11 1/4 × 7 × 7 1/2 in. (2...,NaN,NaN,italiano,NaN,NaN,...,None,NaN,NaN,1540.0,1540.0,True,NaN,NaN,NaN,MET
1204,The Adoration of the Magi,Hugo van der Goes,NaN,https://images.metmuseum.org/CRDImages/ep/orig...,29 1/8 x 25 5/8 in. (74 x 65.1 cm),NaN,NaN,neerlandés,NaN,NaN,...,madera,NaN,NaN,NaN,NaN,False,65.1,74.0,NaN,MET
1205,A City on a Rock,Goya,NaN,https://images.metmuseum.org/CRDImages/ep/orig...,33 × 41 in. (83.8 × 104.1 cm),NaN,NaN,español,NaN,NaN,...,lienzo,NaN,NaN,NaN,NaN,False,104.1,83.8,NaN,MET


In [102]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2918 entries, ('FridaKahlo', 0) to 1206
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   title               2918 non-null   object 
 1   artist              2649 non-null   object 
 2   description         1184 non-null   object 
 3   image_path          2918 non-null   object 
 4   dimensions          2647 non-null   object 
 5   artist_deathplace   182 non-null    object 
 6   artist_birthplace   222 non-null    object 
 7   artist_nationality  1069 non-null   object 
 8   artist_sex          265 non-null    object 
 9   keywords            255 non-null    object 
 10  type                2501 non-null   object 
 11  technique           2201 non-null   object 
 12  support             1867 non-null   object 
 13  theme               393 non-null    object 
 14  country             524 non-null    object 
 15  year_start          2431 non-null   float64


In [ ]:
df1.to_csv("data/obras_draft.csv", index=False)

In [ ]:
df_final.to_csv("data/artworks.csv", index=False)